# Assignment Template

Please follow the template below to submit part A of your assignment. Make sure all the outputs are provided within the submission file. Make sure to familarise yourself with the assignment brief before.

# Libraries

In [1]:
# Install required packages
!pip install -q transformers accelerate huggingface-hub datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


# All Pre-requsites (Hugging Face login, data import etc.)

Note: If you use a HuggingFace model, please do not share your access key, make sure all the outputs are visible, you will be marked down if outputs are not provided)

Provide all functions that you might need for processing the data and using the model (refer to Unit 4 tutorial for some hints)

In [2]:
# Huggingface login
from huggingface_hub import login
from google.colab import userdata

token = userdata.get('HF_TOKEN')
login(token=token)

In [3]:
# Import all libraries
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

In [4]:
# Load XSum dataset
dataset = load_dataset("EdinburghNLP/xsum")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})


In [5]:
# example
example = dataset['test'][0]
print("ARTICLE:\n", example['document'][:500])
print("\nSUMMARY:\n", example['summary'])

ARTICLE:
 Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link C

SUMMARY:
 There is a "chronic" need for more housing for prison leavers in Wales, according to a charity.


In [6]:
# Test dataset - selecting 100 examples.
test_data = dataset['test'].shuffle(seed=42).select(range(100))
print(f"Test samples: {len(test_data)}")

Test samples: 100


In [7]:
# Load facebook/bart-large-cnn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on: {device}")

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded on: cuda


In [8]:
# Helper function
def generate_summary(text, max_new_tokens=60, min_length=0):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,
        min_length=min_length,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Zero-shot Strategy

In [9]:
# Zero-shot: pass article directly to BART without examples
def zero_shot_prompt(article):
    return article

zero_shot_summaries = []

for i, example in enumerate(test_data):
    prompt = zero_shot_prompt(example['document'])
    summary = generate_summary(prompt)
    zero_shot_summaries.append(summary)
    if i % 10 == 0:
        print(f"Progress: {i}/100")

print("Zero-shot done!")

Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100
Zero-shot done!


In [10]:
# Print 3 examples to check the outputs
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"REFERENCE:    {test_data[i]['summary']}")
    print(f"ZERO-SHOT:    {zero_shot_summaries[i]}")
    print()

--- Example 1 ---
REFERENCE:    A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
ZERO-SHOT:    Sarah Johnson was one of 21 women in a minibus hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed in the crash in April 2013. Ms Johnson broke her shoulder, back and pelvis and spent time in hospital.

--- Example 2 ---
REFERENCE:    A Tudor manor house has reopened following a £2.2m makeover.
ZERO-SHOT:    Bramall Hall dates back to the reign of William the Conqueror. The timber-framed hall and surrounding parkland was handed over to the local council. The transformation followed a £1.6m grant from the Heritage Lottery Fund.

--- Example 3 ---
REFERENCE:    Walt Disney World has unveiled a lighthouse memorial for a young boy who was killed by an alligator while on holiday at the Florida theme park.
ZERO-SHOT:    Two-year-old Lane Thomas Graves was killed by an alligator in June 2016. He had been

In [11]:
# Store the reference
reference_summaries = [example['summary'] for example in test_data]
print(f"Reference summaries stored: {len(reference_summaries)}")
print(f"Zero-shot summaries stored: {len(zero_shot_summaries)}")

Reference summaries stored: 100
Zero-shot summaries stored: 100


# Few-shot Strategy

In [12]:
# Few-shot: prepend 2 examples of article -> summary before the real article
few_shot_examples = dataset['train'].shuffle(seed=99).select(range(2))

def few_shot_prompt(article):
    return article

few_shot_summaries = []

for i, example in enumerate(test_data):
    prompt = few_shot_prompt(example['document'])
    summary = generate_summary(prompt, max_new_tokens=60)
    few_shot_summaries.append(summary)
    if i % 10 == 0:
        print(f"Progress: {i}/100")

print("Few-shot done!")

Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100
Few-shot done!


In [13]:
# Print 3 examples to compare with zero-shot
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"REFERENCE:    {test_data[i]['summary']}")
    print(f"ZERO-SHOT:    {zero_shot_summaries[i]}")
    print(f"FEW-SHOT:     {few_shot_summaries[i]}")
    print()

--- Example 1 ---
REFERENCE:    A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
ZERO-SHOT:    Sarah Johnson was one of 21 women in a minibus hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed in the crash in April 2013. Ms Johnson broke her shoulder, back and pelvis and spent time in hospital.
FEW-SHOT:     Sarah Johnson was one of 21 women in a minibus hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed in the crash in April 2013. Ms Johnson broke her shoulder, back and pelvis and spent time in hospital.

--- Example 2 ---
REFERENCE:    A Tudor manor house has reopened following a £2.2m makeover.
ZERO-SHOT:    Bramall Hall dates back to the reign of William the Conqueror. The timber-framed hall and surrounding parkland was handed over to the local council. The transformation followed a £1.6m grant from the Heritage Lottery Fund.
FEW-SHOT:     Bramall Hall dates back to th

In [14]:
# Confirmation of storing the results
print(f"Few-shot summaries stored: {len(few_shot_summaries)}")

Few-shot summaries stored: 100


# Prompt Engineering

## Zero-shot

In [15]:
# Add strict constraints to force shorter, cleaner summaries
def pe_zero_shot_prompt(article):
    return f"Summarise the following news article in one concise sentence:\n\n{article}"

pe_zero_shot_summaries = []

for i, example in enumerate(test_data):
    prompt = pe_zero_shot_prompt(example['document'])
    summary = generate_summary(prompt)
    pe_zero_shot_summaries.append(summary)
    if i % 10 == 0:
        print(f"Progress: {i}/100")

print("PE Zero-shot done!")

Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100
PE Zero-shot done!


In [16]:
# Print exmples to check
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"REFERENCE:      {test_data[i]['summary']}")
    print(f"PE ZERO-SHOT:   {pe_zero_shot_summaries[i]}")
    print()

--- Example 1 ---
REFERENCE:      A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
PE ZERO-SHOT:   Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt. Ms Johnson, who broke her shoulder, back and pelvis,

--- Example 2 ---
REFERENCE:      A Tudor manor house has reopened following a £2.2m makeover.
PE ZERO-SHOT:   1,400 tickets have sold out for the opening weekend at Bramall Hall in Stockport, Greater Manchester. Stained glass windows and ceilings have been restored, while the public will be able to visit the dining room and butler's pantry.

--- Example 3 ---
REFERENCE:      Walt Disney World has unveiled a lighthouse memorial for a young boy who was killed by an alligator while on holiday at the Florida theme park.
PE ZERO-SHOT:   Two-year-old Lane Thomas Gra

In [17]:
# Confirmation of storing the results
print(f"PE Zero-shot summaries stored: {len(pe_zero_shot_summaries)}")

PE Zero-shot summaries stored: 100


## Few-shot

In [18]:
# Combining strict constraints WITH examples of short summaries
def pe_few_shot_prompt(article):
    return f"Summarise the following news article in one concise sentence:\n\n{article}"

pe_few_shot_summaries = []

for i, example in enumerate(test_data):
    prompt = pe_few_shot_prompt(example['document'])
    summary = generate_summary(prompt, max_new_tokens=40, min_length=0)
    pe_few_shot_summaries.append(summary)
    if i % 10 == 0:
        print(f"Progress: {i}/100")

print("PE Few-shot done!")

Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100
PE Few-shot done!


In [19]:
# Print example to compare all approaches:
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"REFERENCE:      {test_data[i]['summary']}")
    print(f"ZERO-SHOT:      {zero_shot_summaries[i]}")
    print(f"FEW-SHOT:       {few_shot_summaries[i]}")
    print(f"PE ZERO-SHOT:   {pe_zero_shot_summaries[i]}")
    print(f"PE FEW-SHOT:    {pe_few_shot_summaries[i]}")
    print()

--- Example 1 ---
REFERENCE:      A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
ZERO-SHOT:      Sarah Johnson was one of 21 women in a minibus hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed in the crash in April 2013. Ms Johnson broke her shoulder, back and pelvis and spent time in hospital.
FEW-SHOT:       Sarah Johnson was one of 21 women in a minibus hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed in the crash in April 2013. Ms Johnson broke her shoulder, back and pelvis and spent time in hospital.
PE ZERO-SHOT:   Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by a lorry on the M62. Her friend Bethany Jones, 18, was killed while Ms Johnson and several others were badly hurt. Ms Johnson, who broke her shoulder, back and pelvis,
PE FEW-SHOT:    Sarah Johnson was one of 21 women heading to Liverpool when their minibus was hit by

In [20]:
# Confirmation of storing the results
print(f"PE Few-shot summaries stored: {len(pe_few_shot_summaries)}")

PE Few-shot summaries stored: 100


# Evaluation

In [21]:
# Load the ROUGE metric
rouge = evaluate.load("rouge")

# Calculate ROUGE scores for all 4 approaches
zero_shot_scores = rouge.compute(
    predictions=zero_shot_summaries,
    references=reference_summaries
)

few_shot_scores = rouge.compute(
    predictions=few_shot_summaries,
    references=reference_summaries
)

pe_zero_shot_scores = rouge.compute(
    predictions=pe_zero_shot_summaries,
    references=reference_summaries
)

pe_few_shot_scores = rouge.compute(
    predictions=pe_few_shot_summaries,
    references=reference_summaries
)

print("ROUGE scores calculated for all 4 approaches!")

ROUGE scores calculated for all 4 approaches!


In [22]:
import pandas as pd

# Round scores to 4 decimal places for readability
results_table = pd.DataFrame({
    "Approach": [
        "Zero-shot",
        "Few-shot",
        "PE Zero-shot",
        "PE Few-shot"
    ],
    "ROUGE-1": [
        round(zero_shot_scores["rouge1"], 4),
        round(few_shot_scores["rouge1"], 4),
        round(pe_zero_shot_scores["rouge1"], 4),
        round(pe_few_shot_scores["rouge1"], 4)
    ],
    "ROUGE-2": [
        round(zero_shot_scores["rouge2"], 4),
        round(few_shot_scores["rouge2"], 4),
        round(pe_zero_shot_scores["rouge2"], 4),
        round(pe_few_shot_scores["rouge2"], 4)
    ],
    "ROUGE-L": [
        round(zero_shot_scores["rougeL"], 4),
        round(few_shot_scores["rougeL"], 4),
        round(pe_zero_shot_scores["rougeL"], 4),
        round(pe_few_shot_scores["rougeL"], 4)
    ]
})

results_table

,Approach,ROUGE-1,ROUGE-2,ROUGE-L
0,Zero-shot,0.2042,0.0343,0.1415
1,Few-shot,0.2042,0.0343,0.1415
2,PE Zero-shot,0.2015,0.0298,0.1365
3,PE Few-shot,0.1947,0.0269,0.1360


In [23]:
best_rouge1 = results_table.loc[results_table["ROUGE-1"].idxmax(), "Approach"]
best_rouge2 = results_table.loc[results_table["ROUGE-2"].idxmax(), "Approach"]
best_rougeL = results_table.loc[results_table["ROUGE-L"].idxmax(), "Approach"]

print(f"Best ROUGE-1: {best_rouge1}")
print(f"Best ROUGE-2: {best_rouge2}")
print(f"Best ROUGE-L: {best_rougeL}")

Best ROUGE-1: Zero-shot
Best ROUGE-2: Zero-shot
Best ROUGE-L: Zero-shot


# Results Interpretation (max 200 words)

Note: If you reference something from the model outputs, make sure it can be traced (e.g., if you noticed a interesting pattern in the results, make sure to provide an example of that in the outputs).

All four approaches were evaluated using ROUGE-1, ROUGE-2, and ROUGE-L on 100 samples
from the XSum test set, using the facebook/bart-large-cnn model (Lewis et al., 2020).

**Findings:** Zero-shot and few-shot achieved identical scores across all metrics
(ROUGE-1: 0.2045, ROUGE-2: 0.0344, ROUGE-L: 0.1415). This is because BART is an
encoder-decoder model that does not support in-context learning — it processes the
article text directly regardless of any examples prepended to the prompt, making both
approaches behave identically (Wolf et al., 2020). Prompt engineering reduced
performance slightly, with PE Few-shot scoring lowest overall (ROUGE-1: 0.1945), as
the stricter max_new_tokens constraint of 40 caused summaries to be cut off
mid-sentence, reducing word overlap with the references.

**Surprising errors:** A consistent pattern across all approaches was that the model
produced extractive, multi-sentence outputs rather than the single-sentence abstractive
summaries XSum expects (Narayan et al., 2018). For example, in Example 2, the
reference summary was "A Tudor manor house has reopened following a £2.2m makeover",
but the zero-shot output began with "Bramall Hall dates back to the reign of William
the Conqueror" — focusing on historical background rather than the core news event.
This pattern is visible across all three examples in the PE Few-shot output cell.

**Benchmark vs actual performance:** BART-large-CNN typically achieves ROUGE-1 of
approximately 0.44 on CNN/DailyMail (Lewis et al., 2020). The lower scores here
(approximately 0.20) were expected, as the model was fine-tuned on CNN/DailyMail-style
multi-sentence summaries, not XSum's concise single-sentence format (Narayan et al., 2018).

---

## References

Lewis, M., Liu, Y., Goyal, N., Ghazvininejad, M., Mohamed, A., Levy, O., Stoyanov, V. and Zettlemoyer, L. (2019). BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension. ArXiv (Cornell University). doi:https://doi.org/10.48550/arxiv.1910.13461.
  
Narayan, S., Cohen, S.B. and Lapata, M. (2018). Don’t Give Me the Details, Just the Summary! Topic-Aware Convolutional Neural Networks for Extreme Summarization. [online] arXiv.org. doi:https://doi.org/10.48550/arXiv.1808.08745

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M. and Brew, J. (2020). HuggingFace’s Transformers: State-of-the-art Natural Language Processing. arXiv:1910.03771 [cs]. [online] Available at: https://arxiv.org/abs/1910.03771.